In [45]:
import json
import os
import numpy as np
from dataclasses import dataclass, field
from pathlib import Path

import torch
import torch.nn as nn
from torch.utils.data import DataLoader

import utaeT
from utils import iterate, overall_performance, save_results, prepare_output, pad_collate, get_ntrainparams, Config, TreeDataset

In [46]:
@dataclass
class ConfigTestPaths:
    # Dataset / paths
    config_path: Path = field(default_factory=lambda: Path.cwd().parents[4] / "ideas-dslab-group1-shared" / "results_modelT" / "conf.json")
    model_dir: Path = field(default_factory=lambda: Path.cwd().parents[4] / "ideas-dslab-group1-shared" / "results_modelT")
    res_dir: Path = field(default_factory=lambda: Path.cwd().parents[4] / "ideas-dslab-group1-shared" / "results_test_modelT" / "barcelona")

    path_base: Path = Path.cwd().parents[4] / "ideas-dslab-group1-shared" / "preprocessed_data"
    paths_sentinel_data: list = None
    paths_tree_data: list = None
    paths_date_data: list = None
    def __post_init__(self):
        self.paths_sentinel_data = [
            self.path_base / "satellite_patches_bar.npy",
            #self.path_base / "satellite_patches_ham.npy",
            #self.path_base / "satellite_patches_par.npy",
            #self.path_base / "satellite_patches_vie.npy",
        ]
        self.paths_tree_data = [
            self.path_base / "tree_patches_bar.npy",
            #self.path_base / "tree_patches_ham.npy",
            #self.path_base / "tree_patches_par.npy",
            #self.path_base / "tree_patches_vie.npy",
        ]
        self.paths_date_data = [
            self.path_base / "date_patches_bar.npy",
            #self.path_base / "date_patches_ham.npy",
            #self.path_base / "date_patches_par.npy",
            #self.path_base / "date_patches_vie.npy",
        ]

In [47]:
@dataclass
class ConfigTest:
    # computation and display
    display_step: int = 1
    num_workers: int = 4
    batch_size: int = 4
    device: str = field(default_factory=lambda: "cuda" if torch.cuda.is_available() else "cpu")
    def torch_device(self):
        return torch.device(self.device)
    fold: int = 1


In [48]:
def load_config(config_test):
    # load the config used for training
    with open(config_test.config_path, "r") as f:
        cfg_dict = json.load(f)

    config_train = Config()

    for k, v in cfg_dict.items():
        if hasattr(config_train, k):
            try:
                if isinstance(getattr(config_train, k), Path):
                    v = Path(v)
            except:
                pass
            setattr(config_train, k, v)

    return config_train

In [49]:
def main(config_train, config_test,config_test_paths):
    # set random seed and device
    np.random.seed(config_train.rdm_seed)
    torch.manual_seed(config_train.rdm_seed)
    device = torch.device(config_test.device)

    sentinel_list = []
    tree_list = []
    date_list = []
    
    for p_s, p_t, p_d in zip(config_test_paths.paths_sentinel_data,config_test_paths.paths_tree_data,config_test_paths.paths_date_data):
        sentinel_list.append(np.load(p_s, mmap_mode='r'))
        tree_list.append(np.load(p_t, mmap_mode='r'))
        date_list.append(np.load(p_d, mmap_mode='r'))

    # merge all cities
    test_x = np.concatenate(sentinel_list, axis=0)
    test_y = np.concatenate(tree_list, axis=0)
    test_d = np.concatenate(date_list, axis=0)

    dt_test  = TreeDataset(test_x, test_d, test_y)

    # create dataloaders using padding
    collate_fn = lambda x: pad_collate(x, pad_value=config_train.pad_value)
    test_loader = DataLoader(
        dt_test,
        batch_size=config_test.batch_size,
        shuffle=False,
        drop_last=False,
        collate_fn=collate_fn,
    )

    print(f"Test {len(dt_test)}")

    # load model structure
    model = utaeT.UTAE(config_train)
    model = model.to(device)

    config_test.N_params = get_ntrainparams(model)
    print(model)
    print("TOTAL TRAINABLE PARAMETERS :", config_test.N_params)

    # create lists for test evaluation
    all_preds_folds = []
    all_targets_folds = []

    # define which folds should be tested
    if config_test.fold is not None:
        folds_to_run = [config_test.fold - 1]
    else:
        folds_to_run = range(5)

    # prepare output folders
    prepare_output(config_test_paths,config_test)

    for fold in folds_to_run:
        # Load weights
        sd = torch.load(
            os.path.join(config_test_paths.model_dir, "Fold_{}".format(fold+1), "model.pth.tar"),
            map_location=device,
        )
        model.load_state_dict(sd["state_dict"])

        # Loss
        criterion = nn.SmoothL1Loss()
    
        print(f"\n=== Running Fold {fold+1} ===")
        
        # Inference
        print("Testing . . .")
        model.eval()
        test_metrics, preds, targets, imgs = iterate(
            model,
            data_loader=test_loader,
            criterion=criterion,
            config=config_test,
            optimizer=None,
            mode="test",
            img_output=True
        )
        all_preds_folds.extend(preds)
        all_targets_folds.extend(targets)
        print(f"Loss: {test_metrics['test_loss']:.4f}")
        overall_performance(preds,targets)
        save_results(fold + 1, test_metrics, config_test,config_test_paths,preds, targets, imgs)

    if config_test.fold is None:
        overall_performance(all_preds_folds, all_targets_folds)

In [50]:
if __name__ == "__main__":
    config_test = ConfigTest()
    config_test_paths = ConfigTestPaths()
    config_train = load_config(config_test_paths)

    main(config_train,config_test,config_test_paths)

Test 78
UTAE(
  (in_conv): ConvBlock(
    (conv): ConvLayer(
      (conv): Sequential(
        (0): Conv2d(5, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), padding_mode=reflect)
        (1): GroupNorm(4, 32, eps=1e-05, affine=True)
        (2): ReLU()
        (3): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), padding_mode=reflect)
        (4): GroupNorm(4, 32, eps=1e-05, affine=True)
        (5): ReLU()
      )
    )
  )
  (down_blocks): ModuleList(
    (0): DownConvBlock(
      (down): ConvLayer(
        (conv): Sequential(
          (0): Conv2d(32, 32, kernel_size=(4, 4), stride=(2, 2), padding=(1, 1), padding_mode=reflect)
          (1): GroupNorm(4, 32, eps=1e-05, affine=True)
          (2): ReLU()
        )
      )
      (conv1): ConvLayer(
        (conv): Sequential(
          (0): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), padding_mode=reflect)
          (1): GroupNorm(4, 32, eps=1e-05, affine=True)
          (2): ReLU()
      